<a href="https://colab.research.google.com/github/khadkaabishek/GEC_Case-Study/blob/main/GEC_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grammatical Error Correction (GEC) Case Study

This notebook demonstrates how to load and use a fine-tuned T5 (Text-to-Text Transfer Transformer) model from Hugging Face's `transformers` library to perform Grammatical Error Correction (GEC). GEC is treated here as a translation task: translating from "incorrect English" to "correct English".

## 1. Installation

In [ ]:
!pip install -q transformers datasets torch sentencepiece ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.4 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check if GPU is available to speed up inference
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 3. Load the Model and Tokenizer

In [ ]:
# We use a T5-base model specifically fine-tuned for Grammatical Error Correction (GEC)
model_name = "vennify/t5-base-grammar-correction"

print("Loading model and tokenizer... This might take a minute.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("Model loaded successfully!")

Loading model and tokenizer... This might take a minute.


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Model loaded successfully!


## 4. Define the Correction Function

In [ ]:
def correct_grammar(text, max_length=128):
    """
    Takes an ungrammatical sentence, adds the required task prefix for T5,
    and returns the grammatically corrected version.
    """
    # T5 models expect a prefix indicating the task
    input_text = f"gec: {text}"

    # Tokenize the input text
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    ).to(device)

    # Generate the corrected tokens
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,        # Beam search helps find higher quality sequences
            early_stopping=True
        )

    # Decode the tokens back into human-readable text
    corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_text

## 5. Test and Evaluation

In [ ]:
# Let's test the pipeline on a mix of common tense, preposition, and agreement errors.
test_sentences = [
    "He do not went to school yesterday.",
    "I am looking forward to meet you next week.",
    "She have two cat and an dog.",
    "The data show that we are making progress.",
    "Every students must brings their own book."
]

print("--- GEC Model Evaluation --- \n")
for sentence in test_sentences:
    corrected = correct_grammar(sentence)
    print(f"Original : {sentence}")
    print(f"Corrected: {corrected}")
    print("-" * 40)

--- GEC Model Evaluation --- 

Original : He do not went to school yesterday.
Corrected: He did not go to school yesterday.
----------------------------------------
Original : I am looking forward to meet you next week.
Corrected: I am looking forward to meeting you next week.
----------------------------------------
Original : She have two cat and an dog.
Corrected: She has two cats and a dog.
----------------------------------------
Original : The data show that we are making progress.
Corrected: The data show that we are making progress.
----------------------------------------
Original : Every students must brings their own book.
Corrected: Every student must bring their own book.
----------------------------------------


In [ ]:
import ipywidgets as widgets
# Cell 5: UI Interface Configuration
# 1. Create interactive UI elements
input_box = widgets.Text(
    value='',
    placeholder='Type your sentence here (e.g., He do not went to school)...',
    description='Input Text:',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description='Correct Grammar',
    button_style='success',
    tooltip='Click to run NLP correction engine',
    icon='check'
)

output_area = widgets.Output()

# 2. Define click behavior
def on_button_clicked(b):
    with output_area:
        # clear_output() # Clears previous results
        user_input = input_box.value

        print("⚡ Processing text through model...")
        corrected_result = correct_grammar(user_input)

        # Display results cleanly
        print("\n" + "="*50)
        print(f"❌ Original : {user_input}")
        print(f"   Corrected: {corrected_result}")
        print("="*50 + "\n")

# Connect the button to the function logic
submit_button.on_click(on_button_clicked)

# 3. Render the UI
print("👉 Type a sentence below and click the button:")
display(widgets.HBox([input_box, submit_button]))
display(output_area)

👉 Type a sentence below and click the button:


Output()